# Fine-tuning a masked language model

## Load Dataset:

In [1]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

## Load Tokenizer:

In [2]:
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(ckpt)

## Load Model:

In [3]:
from transformers import AutoModelForMaskedLM

ckpt = "distilbert-base-uncased"

model = AutoModelForMaskedLM.from_pretrained(ckpt)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## Tokenize Dataset:

In [4]:
tokenized_datasets = raw_datasets.map(
    function=lambda x: tokenizer(x['text'], truncation=True, max_length=512), 
    batched=True, 
    remove_columns=["text", "label"]
)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

## Data Collator:

In [5]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

## Domain Adapt Model:

In [10]:
from transformers import TrainingArguments
from transformers import Trainer

args = TrainingArguments(
    output_dir="distilbert-mlm-imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=lambda x: {"perplexity": math.exp(x.loss)},
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 